In [1]:
# 한글 토크나이저 다운로드
!pip install numpy==1.26.4
!pip install matplotlib
!pip install kiwipiepy
!pip install ipykernel
!pip install torch

In [2]:
# 글 확인하기 
proverbs = []
with open('proverbs.txt', 'r', encoding='utf-8') as f:
    for line in f:
        proverbs.append(line.strip())

print(proverbs[0]) # 첫 번째 속담 확인



가갸 뒷다리도 모른다


In [3]:
#한글 토크나이저 사용
from kiwipiepy import Kiwi

kiwi = Kiwi()

text = proverbs[0]

# 형태소 분석
tokens = kiwi.tokenize(text)

for token in tokens:
    print(token.form, token.tag)

##

가갸 NNG
뒷다리 NNG
도 JX
모르 VV
ᆫ다 EF


In [4]:
from kiwipiepy import Kiwi

kiwi = Kiwi()

# Word2Vec 학습에 사용할 품사
target_tags = {
    'NNG',  # 일반 명사
    'NNP',  # 고유 명사
    'VV',   # 동사
    'VA',   # 형용사
    'MAG',  # 일반 부사
    'MM',   # 관형사
    'NR'    # 수사
}

tokenized_proverbs = []

for proverb in proverbs:
    tokens = kiwi.tokenize(proverb)

    words = [
        token.form
        for token in tokens
        if token.tag in target_tags
    ]

    tokenized_proverbs.append(words)

In [5]:

import numpy as np

min_count = 3

# 1. 전체 속담의 단어를 하나로 모으기 (빈도 계산용)
words = [
    word
    for proverb in tokenized_proverbs
    for word in proverb
]

# 2. 단어별 출현 빈도 계산
counts = {}

for word in words:
    counts[word] = counts.get(word, 0) + 1

# 3. 최소 출현 빈도 적용
filtered_proverbs = []

for proverb in tokenized_proverbs:
    filtered_words = [
        word if counts[word] >= min_count else '<unk>'
        for word in proverb
    ]
    filtered_proverbs.append(filtered_words)

# 4. 단어 사전 생성
word_to_id = {}
id_to_word = {}

for proverb in filtered_proverbs:
    for word in proverb:
        if word not in word_to_id:
            new_id = len(word_to_id)
            word_to_id[word] = new_id
            id_to_word[new_id] = word

# 5. 속담별 Corpus 생성
corpus = [
    np.array([word_to_id[word] for word in proverb])
    for proverb in filtered_proverbs
]

# 전체 단어 ID 배열도 별도로 생성
flat_corpus = np.concatenate(corpus)

# 6. 결과 확인
vocab_size = len(word_to_id)

print('속담 개수:', len(proverbs))
print('말뭉치 크기:', len(flat_corpus))
print('전체 어휘 수:', len(counts))
print('최종 어휘 수:', vocab_size)

print('<unk> 비율: %.1f%%' % (
    100 * np.mean(flat_corpus == word_to_id['<unk>'])
))

# 7. 출현 빈도 상위 40개
sorted_words = sorted(
    counts.items(),
    key=lambda x: x[1],
    reverse=True
)

print('자주 나온 단어:', sorted_words[:40])

# 8. 결과 샘플 확인
print('\n첫 번째 속담:')
print(proverbs[0])
print(filtered_proverbs[0])


속담 개수: 10763
말뭉치 크기: 47278
전체 어휘 수: 5500
최종 어휘 수: 2237
<unk> 비율: 8.9%
자주 나온 단어: [('하', 969), ('먹', 812), ('가', 591), ('없', 572), ('보', 480), ('말', 447), ('한', 434), ('못', 431), ('있', 430), ('죽', 418), ('사람', 365), ('같', 364), ('나', 364), ('개', 352), ('물', 348), ('안', 309), ('되', 293), ('알', 282), ('집', 275), ('남', 269), ('치', 246), ('밥', 229), ('살', 222), ('모르', 214), ('소', 206), ('크', 202), ('떡', 200), ('쓰', 197), ('들', 192), ('오', 185), ('열', 183), ('좋', 174), ('맞', 169), ('날', 165), ('잘', 162), ('똥', 162), ('눈', 158), ('속', 156), ('이', 154), ('자식', 152)]

첫 번째 속담:
가갸 뒷다리도 모른다
['<unk>', '뒷다리', '모르']


In [6]:
import numpy as np

def create_contexts_target(corpus, window_size):
    contexts = []
    target = []

    # 속담별로 독립적으로 Context 생성
    for sentence in corpus:
        for idx in range(window_size, len(sentence) - window_size):

            target.append(sentence[idx])

            context = []
            for offset in range(-window_size, window_size + 1):
                if offset == 0:
                    continue
                context.append(sentence[idx + offset])

            contexts.append(context)

    return np.array(contexts), np.array(target)


window_size = 3

contexts, target = create_contexts_target(corpus, window_size)

print('Context 크기:', contexts.shape)
print('Target 크기:', target.shape)

Context 크기: (2395, 6)
Target 크기: (2395,)


In [7]:

import torch
import torch.nn as nn

class CBOW(nn.Module):
    def __init__(self, vocab_size, hidden_size, window_size):
        super().__init__()

        # 단어 ID -> Embedding Vector
        self.embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        # Hidden -> Vocabulary 크기
        self.output = nn.Linear(
            hidden_size,
            vocab_size
        )

    def forward(self, contexts):
        # contexts: (batch_size, 2 * window_size)

        # 각 주변 단어의 Vector 추출
        embeds = self.embedding(contexts)

        # 주변 단어 Vector 평균
        h = embeds.mean(dim=1)

        # 중심 단어 예측
        scores = self.output(h)

        return scores


In [8]:

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

# GPU 설정
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print('사용 장치:', device)

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# 하이퍼파라미터
hidden_size = 100
batch_size = 100
max_epoch = 30
learning_rate = 0.001

# NumPy -> PyTorch Tensor
contexts_tensor = torch.tensor(
    contexts, dtype=torch.long
)

target_tensor = torch.tensor(
    target, dtype=torch.long
)

# Dataset 및 DataLoader
dataset = TensorDataset(
    contexts_tensor,
    target_tensor
)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True
)

# 모델 생성
model = CBOW(
    vocab_size,
    hidden_size,
    window_size
).to(device)

# 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

# 학습
loss_history = []

for epoch in range(max_epoch):

    model.train()
    total_loss = 0

    for batch_contexts, batch_target in dataloader:

        batch_contexts = batch_contexts.to(device)
        batch_target = batch_target.to(device)

        # Forward
        output = model(batch_contexts)

        # Loss
        loss = criterion(output, batch_target)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(batch_target)

    avg_loss = total_loss / len(dataset)
    loss_history.append(avg_loss)

    print(
        f'Epoch {epoch+1:02d}/{max_epoch} '
        f'Loss: {avg_loss:.4f}'
    )

print('학습 완료')


/home/jinju/ssu/2026-2/deep_learning/word2vec/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


사용 장치: cpu
Epoch 01/30 Loss: 7.6681
Epoch 02/30 Loss: 7.3651
Epoch 03/30 Loss: 7.0687
Epoch 04/30 Loss: 6.7554
Epoch 05/30 Loss: 6.4333
Epoch 06/30 Loss: 6.1127
Epoch 07/30 Loss: 5.8093
Epoch 08/30 Loss: 5.5322
Epoch 09/30 Loss: 5.2884
Epoch 10/30 Loss: 5.0768
Epoch 11/30 Loss: 4.8892
Epoch 12/30 Loss: 4.7208
Epoch 13/30 Loss: 4.5676
Epoch 14/30 Loss: 4.4246
Epoch 15/30 Loss: 4.2869
Epoch 16/30 Loss: 4.1560
Epoch 17/30 Loss: 4.0279
Epoch 18/30 Loss: 3.9035
Epoch 19/30 Loss: 3.7819
Epoch 20/30 Loss: 3.6629
Epoch 21/30 Loss: 3.5469
Epoch 22/30 Loss: 3.4340
Epoch 23/30 Loss: 3.3233
Epoch 24/30 Loss: 3.2155
Epoch 25/30 Loss: 3.1106
Epoch 26/30 Loss: 3.0087
Epoch 27/30 Loss: 2.9098
Epoch 28/30 Loss: 2.8139
Epoch 29/30 Loss: 2.7202
Epoch 30/30 Loss: 2.6307
학습 완료


In [9]:
# 개선된 Word2Vec: SGNS + 희귀어 삭제 + 빈도 subsampling
# 기존 CBOW 셀은 개념 설명용으로 남겨두고, 실제 검색은 아래 모델을 사용합니다.
import random
from collections import Counter

random.seed(42); np.random.seed(42); torch.manual_seed(42)

# Kiwi의 VV/VA token.form은 이미 어간이다. 예: 피다 -> '피' (피-)
# 품사 표기를 보존하기 위해 동사/형용사 어간에는 '-'를 붙인다.
sentences = []
for proverb in proverbs:
    sent = []
    for t in kiwi.tokenize(proverb):
        if t.tag in target_tags:
            sent.append(t.form + ('-' if t.tag in {'VV', 'VA'} else ''))
    sentences.append(sent)

min_count = 2
freq = Counter(w for s in sentences for w in s)
sentences = [[w for w in s if freq[w] >= min_count] for s in sentences]
vocab = sorted({w for s in sentences for w in s})
word_to_id = {w:i for i,w in enumerate(vocab)}; id_to_word = dict(enumerate(vocab))
corpus = [[word_to_id[w] for w in s] for s in sentences]
counts = np.array([freq[w] for w in vocab], dtype=np.float64)

# 너무 흔한 기능어가 모든 문맥을 지배하지 않도록 subsampling
total = counts.sum(); t = 1e-3
keep_prob = np.minimum(1.0, (np.sqrt(counts / total / t) + 1) * (t / (counts / total)))
rng = np.random.default_rng(42)
corpus = [[w for w in s if rng.random() < keep_prob[w]] for s in corpus]

window_size = 4
pairs = []
for s in corpus:
    for i, center in enumerate(s):
        radius = rng.integers(1, window_size + 1)
        for j in range(max(0, i-radius), min(len(s), i+radius+1)):
            if i != j: pairs.append((center, s[j]))
centers = torch.tensor([p[0] for p in pairs], dtype=torch.long)
contexts = torch.tensor([p[1] for p in pairs], dtype=torch.long)

class SGNS(nn.Module):
    def __init__(self, n, d):
        super().__init__(); self.in_embed=nn.Embedding(n,d); self.out_embed=nn.Embedding(n,d)
    def forward(self, c, o, neg):
        score = (self.in_embed(c) * self.out_embed(o)).sum(1)
        neg_score = torch.bmm(self.out_embed(neg), self.in_embed(c).unsqueeze(2)).squeeze(2)
        return -torch.log(torch.sigmoid(score)+1e-9).mean() - torch.log(torch.sigmoid(-neg_score)+1e-9).mean()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sgns = SGNS(len(vocab), 100).to(device); opt = torch.optim.Adam(sgns.parameters(), lr=0.003)
noise = torch.tensor(counts ** 0.75 / (counts ** 0.75).sum(), dtype=torch.float32, device=device)
batch_size, negatives, epochs = 512, 10, 80
for epoch in range(epochs):
    order = torch.randperm(len(pairs))
    loss_sum = 0.0
    for ix in order.split(batch_size):
        c, o = centers[ix].to(device), contexts[ix].to(device)
        neg = torch.multinomial(noise, len(ix)*negatives, replacement=True).view(len(ix), negatives)
        loss = sgns(c, o, neg); opt.zero_grad(); loss.backward(); opt.step(); loss_sum += loss.item()*len(ix)
    if (epoch+1) % 10 == 0: print(f'SGNS epoch {epoch+1:02d}/{epochs}, loss={loss_sum/len(pairs):.4f}')

# 검색은 SGNS 벡터 단독으로 하지 않는다.
# 작은 속담 말뭉치에서는 SGNS가 우연한 근접 단어를 만들 수 있기 때문이다.
# 실제 속담 문맥의 PPMI 벡터를 사용하면 공통 문맥이 없는 단어의 점수는 0이다.
cooc = np.zeros((len(vocab), len(vocab)), dtype=np.float32)
for s in corpus:
    for i, center in enumerate(s):
        for j in range(max(0, i-window_size), min(len(s), i+window_size+1)):
            if i != j: cooc[center, s[j]] += 1
row_sum = cooc.sum(axis=1, keepdims=True); col_sum = cooc.sum(axis=0, keepdims=True); total_cooc = cooc.sum()
expected = row_sum @ col_sum / max(total_cooc, 1)
ppmi = np.maximum(np.log((cooc * total_cooc + 1e-9) / (expected + 1e-9)), 0)
ppmi[cooc == 0] = 0
ppmi_embeddings = torch.tensor(ppmi, dtype=torch.float32)

def most_similar(word, k=10):
    if word not in word_to_id: print(f"'{word}' 는(은) 사전에 없음"); return
    qid = word_to_id[word]; q = ppmi_embeddings[qid].unsqueeze(0)
    sim = torch.nn.functional.cosine_similarity(ppmi_embeddings, q, dim=1)
    # 공통 문맥이 하나도 없는 후보는 제거
    shared = ((ppmi_embeddings > 0) & (q > 0)).sum(dim=1)
    sim[shared == 0] = -float('inf'); sim[qid] = -float('inf')
    print(f"'{word}' 와 가장 유사한 단어 Top {k}:")
    for i in torch.topk(sim, k).indices: print(f'{id_to_word[int(i)]}: {sim[int(i)]:.4f}')

most_similar('꽃'); most_similar('개')

SGNS epoch 10/80, loss=2.2028
SGNS epoch 20/80, loss=0.8140
SGNS epoch 30/80, loss=0.4987
SGNS epoch 40/80, loss=0.3901
SGNS epoch 50/80, loss=0.3406
SGNS epoch 60/80, loss=0.3166
SGNS epoch 70/80, loss=0.3018
SGNS epoch 80/80, loss=0.2953
'꽃' 와 가장 유사한 단어 Top 10:
피-: 0.5784
나비: 0.2932
열매: 0.2629
돋치-: 0.2408
향기: 0.2282
돌: 0.2206
붉-: 0.2182
고목: 0.2150
나무: 0.2136
꺾이-: 0.2124
'개' 와 가장 유사한 단어 Top 10:
먹-: 0.2789
새끼: 0.2703
하-: 0.2691
호랑이: 0.2646
꼬리: 0.2632
남: 0.2627
똥: 0.2569
안: 0.2556
집: 0.2545
고양이: 0.2526


In [28]:
# SGNS 벡터로 유추: a:b = c:?
word_vecs = sgns.in_embed.weight.detach()

def analogy(a, b, c, word_to_id, id_to_word, word_vecs, top=5):
    for word in (a, b, c):
        if word not in word_to_id:
            print(f"'{word}' 는(은) 사전에 없음")
            return

    # b - a + c
    query = word_vecs[word_to_id[b]] - word_vecs[word_to_id[a]] + word_vecs[word_to_id[c]]
    query = query / (query.norm() + 1e-8)
    normalized = word_vecs / (word_vecs.norm(dim=1, keepdim=True) + 1e-8)
    similarity = normalized @ query

    # 입력으로 사용한 세 단어는 결과에서 제외
    for word in (a, b, c):
        similarity[word_to_id[word]] = -float('inf')

    print(f"{a} : {b} = {c} : ?")
    for idx in torch.topk(similarity, top).indices:
        print(f'{id_to_word[int(idx)]}: {similarity[int(idx)].item():.4f}')

analogy('소', '송아지', '개', word_to_id, id_to_word, word_vecs)


소 : 송아지 = 개 : ?
음지: 0.4501
동서: 0.4173
옆: 0.3900
발가락: 0.3875
끼이-: 0.3860
